# Fine-tune YOLOX-M / ByteTrack on DanceTrack

Notebook này dùng cho container/Jupyter GPU. Dataset DanceTrack được đọc trực tiếp từ mount path, chỉ ghi annotation JSON và checkpoint output vào thư mục làm việc.

Cấu trúc dataset đang hỗ trợ:

```text
dancetrack-zip/
  train1/train1/dancetrack*/img1, gt
  train2/train2/dancetrack*/img1, gt
  val/val/dancetrack*/img1, gt
  test1/test1/...
  test2/test2/...
```


In [ ]:
# Cell 1: Configure paths
from pathlib import Path
import os

REPO_DIR = Path('/workspace/bytetrack')
DATASET_ROOT = Path('/workspace/datasets/dancetrack-zip')
PRETRAINED_CKPT = Path('/workspace/yolox_m_pretrained.pth')

# Neu notebook dang nam trong repo, co the dung thu muc hien tai.
if not REPO_DIR.exists() and Path.cwd().name == 'bytetrack':
    REPO_DIR = Path.cwd()

print('REPO_DIR =', REPO_DIR)
print('DATASET_ROOT =', DATASET_ROOT)
print('PRETRAINED_CKPT =', PRETRAINED_CKPT)

print('REPO exists:', REPO_DIR.exists())
print('DATASET exists:', DATASET_ROOT.exists())
print('CKPT exists:', PRETRAINED_CKPT.exists())

In [ ]:
# Cell 2: Clone repo if needed
import os
from pathlib import Path

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    !git clone https://github.com/khangkaka066/bytetrack.git {REPO_DIR}

os.chdir(REPO_DIR)
print('Current directory:', Path.cwd())


In [ ]:
# Cell 3: Install dependencies
import sys
from pathlib import Path

PY = sys.executable
print('Python executable:', PY)

!{PY} -m pip install -q -r requirements-kaggle.txt
!{PY} -m pip install -q gdown opencv-python-headless

# Khong dung pip install -e . tren Vast vi mot so image khong co lenh python
# hoac build editable bi loi khi torch chua nam trong build env.
repo_str = str(Path.cwd())
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)

import yolox
import cv2
import torch
print('YOLOX import OK:', yolox.__file__)
print('cv2 OK:', cv2.__version__)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
print('gpu count:', torch.cuda.device_count())


In [ ]:
# Cell 4: Prepare checkpoint
import sys
from pathlib import Path

PY = sys.executable

Path('pretrained').mkdir(exist_ok=True)
target_ckpt = Path('pretrained/yolox_m_pretrained.pth')

if PRETRAINED_CKPT.exists():
    if not target_ckpt.exists():
        !cp {PRETRAINED_CKPT} {target_ckpt}
else:
    # Google Drive file ID tu link ban dua.
    !{PY} -m gdown --id 11Zb0NN_Uu7JwUd9e6Nk8o2_EUfxWqsun -O {target_ckpt}

if not target_ckpt.exists() or target_ckpt.stat().st_size == 0:
    raise FileNotFoundError(
        f'Missing checkpoint: {target_ckpt}. Put yolox_m_pretrained.pth at '
        f'{PRETRAINED_CKPT} or rerun the gdown download cell.'
    )

print('Checkpoint ready:', target_ckpt, target_ckpt.stat().st_size, 'bytes')
!ls -lh pretrained


In [ ]:
# Cell 5: Inspect dataset
print('Dataset root exists:', DATASET_ROOT.exists())
!find {DATASET_ROOT} -maxdepth 3 -type d | head -80
!find {DATASET_ROOT} -maxdepth 5 -type d -name img1 | head -20
!find {DATASET_ROOT} -maxdepth 5 -type f -name gt.txt | head -20


In [ ]:
# Cell 6: Write DanceTrack-to-COCO converter
from pathlib import Path

Path('tools').mkdir(exist_ok=True)
converter = r'''
import os
import json
import cv2
import numpy as np
from pathlib import Path

DATASET_ROOT = Path(os.environ['DANCETRACK_ROOT'])
TRAIN_ROOTS = [
    DATASET_ROOT / 'train1' / 'train1',
    DATASET_ROOT / 'train2' / 'train2',
]
VAL_ROOTS = [
    DATASET_ROOT / 'val' / 'val',
]

OUT_DIR = Path('datasets/dancetrack/annotations')
OUT_DIR.mkdir(parents=True, exist_ok=True)


def collect_sequences(roots):
    seqs = []
    for root in roots:
        if not root.exists():
            print('missing:', root)
            continue
        seqs.extend([
            p for p in sorted(root.iterdir())
            if p.is_dir() and (p / 'img1').exists()
        ])
    return seqs


def load_gt(gt_path):
    if not gt_path.exists() or gt_path.stat().st_size == 0:
        return np.empty((0, 9), dtype=np.float32)
    anns = np.loadtxt(str(gt_path), dtype=np.float32, delimiter=',')
    if anns.ndim == 1:
        anns = anns.reshape(1, -1)
    return anns


def convert(split, roots):
    seqs = collect_sequences(roots)
    if len(seqs) == 0:
        raise RuntimeError(f'No sequences found for {split}')

    out = {
        'images': [],
        'annotations': [],
        'videos': [],
        'categories': [{'id': 1, 'name': 'pedestrian'}],
    }

    image_id = 0
    ann_id = 0
    video_id = 0
    global_tid = 0
    tid_map = {}

    for seq_path in seqs:
        seq_name = seq_path.name
        img_dir = seq_path / 'img1'
        gt_path = seq_path / 'gt' / 'gt.txt'

        video_id += 1
        out['videos'].append({'id': video_id, 'file_name': seq_name})

        images = sorted([
            p for p in img_dir.iterdir()
            if p.suffix.lower() in ['.jpg', '.jpeg', '.png']
        ])
        frame_to_image_id = {}

        for frame_id, img_path in enumerate(images, start=1):
            img = cv2.imread(str(img_path))
            if img is None:
                print('skip unreadable image:', img_path)
                continue

            h, w = img.shape[:2]
            image_id += 1
            frame_to_image_id[frame_id] = image_id
            out['images'].append({
                'file_name': str(img_path),
                'id': image_id,
                'frame_id': frame_id,
                'prev_image_id': image_id - 1 if frame_id > 1 else -1,
                'next_image_id': image_id + 1 if frame_id < len(images) else -1,
                'video_id': video_id,
                'height': h,
                'width': w,
            })

        anns = load_gt(gt_path)
        for row in anns:
            frame_id = int(row[0])
            local_tid = int(row[1])
            x, y, bw, bh = row[2:6].tolist()
            if frame_id not in frame_to_image_id or bw <= 0 or bh <= 0:
                continue

            key = (seq_name, local_tid)
            if key not in tid_map:
                global_tid += 1
                tid_map[key] = global_tid

            ann_id += 1
            out['annotations'].append({
                'id': ann_id,
                'category_id': 1,
                'image_id': frame_to_image_id[frame_id],
                'track_id': tid_map[key],
                'bbox': [float(x), float(y), float(bw), float(bh)],
                'conf': 1.0,
                'iscrowd': 0,
                'area': float(bw * bh),
            })

        print(f'{split}: {seq_name} - {len(images)} images')

    out_file = OUT_DIR / f'{split}.json'
    with open(out_file, 'w') as f:
        json.dump(out, f)
    print(f'saved {out_file}: {len(out["images"])} images, {len(out["annotations"])} annotations')


convert('train', TRAIN_ROOTS)
convert('val', VAL_ROOTS)
'''

Path('tools/convert_dancetrack_input_to_coco.py').write_text(converter)
print('Wrote tools/convert_dancetrack_input_to_coco.py')


In [ ]:
# Cell 7: Convert annotations. This does not copy image files.
import os
import sys
os.environ['DANCETRACK_ROOT'] = str(DATASET_ROOT)
os.environ['PYTHONPATH'] = str(REPO_DIR) + os.pathsep + os.environ.get('PYTHONPATH', '')
PY = sys.executable

!{PY} tools/convert_dancetrack_input_to_coco.py
!ls -lh datasets/dancetrack/annotations


In [ ]:
# Cell 8: Patch MOTDataset so absolute image paths are read directly.
from pathlib import Path

p = Path('yolox/data/datasets/mot.py')
s = p.read_text()
old = """        img_file = os.path.join(\n            self.data_dir, self.name, file_name\n        )\n        img = cv2.imread(img_file)\n"""
new = """        if os.path.isabs(file_name):\n            img_file = file_name\n        else:\n            img_file = os.path.join(\n                self.data_dir, self.name, file_name\n            )\n        img = cv2.imread(img_file)\n"""

if old in s:
    p.write_text(s.replace(old, new))
    print('Patched', p)
else:
    print('Patch already applied or source changed')


In [ ]:
# Cell 9: Write YOLOX-M fine-tuning experiment
from pathlib import Path

exp_text = r'''
import os
import torch
import torch.distributed as dist

from yolox.exp import Exp as MyExp
from yolox.data import get_yolox_datadir


class Exp(MyExp):
    def __init__(self):
        super().__init__()

        self.num_classes = 1
        self.depth = 0.67
        self.width = 0.75
        self.exp_name = 'yolox_m_finetune_dancetrack'

        self.dataset_name = 'dancetrack'
        self.train_ann = 'train.json'
        self.val_ann = 'val.json'

        self.input_size = (800, 1440)
        self.test_size = (800, 1440)
        self.random_size = (18, 32)

        self.max_epoch = int(os.environ.get('MAX_EPOCH', '30'))
        self.no_aug_epochs = int(os.environ.get('NO_AUG_EPOCHS', '5'))
        self.warmup_epochs = 1
        self.eval_interval = int(os.environ.get('EVAL_INTERVAL', '5'))
        self.print_interval = 20
        self.basic_lr_per_img = 0.001 / 64.0

        self.test_conf = 0.001
        self.nmsthre = 0.7

    def get_data_loader(self, batch_size, is_distributed, no_aug=False):
        from yolox.data import (
            MOTDataset,
            TrainTransform,
            YoloBatchSampler,
            DataLoader,
            InfiniteSampler,
            MosaicDetection,
        )

        dataset = MOTDataset(
            data_dir=os.path.join(get_yolox_datadir(), self.dataset_name),
            json_file=self.train_ann,
            name='',
            img_size=self.input_size,
            preproc=TrainTransform(
                rgb_means=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
                max_labels=500,
            ),
        )

        dataset = MosaicDetection(
            dataset,
            mosaic=not no_aug,
            img_size=self.input_size,
            preproc=TrainTransform(
                rgb_means=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
                max_labels=1000,
            ),
            degrees=self.degrees,
            translate=self.translate,
            scale=self.scale,
            shear=self.shear,
            perspective=self.perspective,
            enable_mixup=self.enable_mixup,
        )

        self.dataset = dataset

        if is_distributed:
            batch_size = batch_size // dist.get_world_size()

        sampler = InfiniteSampler(len(self.dataset), seed=self.seed if self.seed else 0)
        batch_sampler = YoloBatchSampler(
            sampler=sampler,
            batch_size=batch_size,
            drop_last=False,
            input_dimension=self.input_size,
            mosaic=not no_aug,
        )

        return DataLoader(
            self.dataset,
            num_workers=self.data_num_workers,
            pin_memory=True,
            batch_sampler=batch_sampler,
        )

    def get_eval_loader(self, batch_size, is_distributed, testdev=False):
        from yolox.data import MOTDataset, ValTransform

        valdataset = MOTDataset(
            data_dir=os.path.join(get_yolox_datadir(), self.dataset_name),
            json_file=self.val_ann,
            name='',
            img_size=self.test_size,
            preproc=ValTransform(
                rgb_means=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
        )

        sampler = torch.utils.data.SequentialSampler(valdataset)
        return torch.utils.data.DataLoader(
            valdataset,
            batch_size=batch_size,
            sampler=sampler,
            num_workers=self.data_num_workers,
            pin_memory=True,
        )

    def get_evaluator(self, batch_size, is_distributed, testdev=False):
        from yolox.evaluators import COCOEvaluator

        return COCOEvaluator(
            dataloader=self.get_eval_loader(batch_size, is_distributed, testdev),
            img_size=self.test_size,
            confthre=self.test_conf,
            nmsthre=self.nmsthre,
            num_classes=self.num_classes,
            testdev=testdev,
        )
'''

Path('exps/example/mot').mkdir(parents=True, exist_ok=True)
Path('exps/example/mot/yolox_m_finetune_dancetrack.py').write_text(exp_text)
print('Wrote exps/example/mot/yolox_m_finetune_dancetrack.py')


In [ ]:
# Cell 10: Smoke-test dataloader
import os
import sys
repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
os.environ['PYTHONPATH'] = repo_str + os.pathsep + os.environ.get('PYTHONPATH', '')

from yolox.exp import get_exp

exp = get_exp('exps/example/mot/yolox_m_finetune_dancetrack.py', None)
loader = exp.get_data_loader(batch_size=1, is_distributed=False, no_aug=True)
batch = next(iter(loader))
print('Loader OK')
print(type(batch))


In [ ]:
# Cell 11: Train
# Neu bi OOM, doi BATCH_SIZE thanh 8 hoac 4.
BATCH_SIZE = 16
DEVICES = 4
MAX_EPOCH = 30
NO_AUG_EPOCHS = 5
EVAL_INTERVAL = 5

import os
import sys
import subprocess
from pathlib import Path

ckpt_path = Path('pretrained/yolox_m_pretrained.pth')
if not ckpt_path.exists() or ckpt_path.stat().st_size == 0:
    raise FileNotFoundError(f'Missing checkpoint before train: {ckpt_path}')

env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR) + os.pathsep + env.get('PYTHONPATH', '')
env['MAX_EPOCH'] = str(MAX_EPOCH)
env['NO_AUG_EPOCHS'] = str(NO_AUG_EPOCHS)
env['EVAL_INTERVAL'] = str(EVAL_INTERVAL)

cmd = [
    sys.executable,
    'tools/train.py',
    '-f', 'exps/example/mot/yolox_m_finetune_dancetrack.py',
    '-c', str(ckpt_path),
    '-b', str(BATCH_SIZE),
    '-d', str(DEVICES),
    '--fp16',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, env=env, check=True)


In [ ]:
# Cell 12: Check outputs
!ls -lh YOLOX_outputs/yolox_m_finetune_dancetrack || true
!find YOLOX_outputs/yolox_m_finetune_dancetrack -maxdepth 2 -type f | sort | tail -30 || true
